#### Ingesting Green Taxi Trip Data - April 2026

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from dotenv import load_dotenv
import logging
import os

load_dotenv()

True

In [2]:
log_file = r"/app/data/logs/"
os.makedirs(name = log_file,exist_ok= True)
file_name_green = os.path.join(log_file,'green_taxi_trips.log')
print(file_name_green)

/app/data/logs/green_taxi_trips.log


In [3]:
logger = logging.getLogger(__name__)
logger.propagate = False   # <-- add this
logger.setLevel(logging.DEBUG)
fh = logging.FileHandler(file_name_green,mode = 'w')
logger.addHandler(fh)
formatter = logging.Formatter('[%(asctime)s] %(levelname)s: %(message)s')
fh.setFormatter(formatter)

In [4]:
spark

NameError: name 'spark' is not defined

#### Defining Schema for Green Taxi Trip Data

In [ ]:
green_taxi_schema = '''
                        VendorID INTEGER,
                        lpep_pickup_datetime TIMESTAMP,
                        lpep_dropoff_datetime TIMESTAMP,
                        store_and_fwd_flag STRING,
                        RatecodeID LONG,
                        PULocationID INTEGER,
                        DOLocationID INTEGER,
                        passenger_count LONG,
                        trip_distance DOUBLE,
                        fare_amount DOUBLE,
                        extra DOUBLE,
                        mta_tax DOUBLE,
                        tip_amount DOUBLE,
                        tolls_amount DOUBLE,
                        ehail_fee DOUBLE,
                        improvement_surcharge DOUBLE,
                        total_amount DOUBLE,
                        payment_type LONG,
                        trip_type LONG,
                        congestion_surcharge DOUBLE,
                        cbd_congestion_fee DOUBLE
'''

#### Reading Green Taxi Trip Parquet file

In [ ]:
green_taxi_data = spark.read.option('header',True)\
                                .schema(green_taxi_schema)\
                                .parquet('/app/data/input/green/green_tripdata_2026-04.parquet')


In [ ]:
green_taxi_data.printSchema()

In [ ]:
green_taxi_data.show(5)

#### Log Raw Count

In [ ]:
logger.info(f"Raw Count : {green_taxi_data.count()}")

In [ ]:
green_taxi_data.rdd.getNumPartitions()

#### Data Quality Checks and Filtering

##### Checking tip amount > 0 for cash payments --> not a valid case [drop them]

In [ ]:

green_taxi_data = green_taxi_data.\
        filter(~((col('payment_type') == 2) & (col('tip_amount').cast("decimal") > 0.0)))

##### Filtering any row where pick up date is greater than 2026-04

In [ ]:
green_taxi_data = green_taxi_data.where(~(date_format('lpep_pickup_datetime',format = 'yyyy-MM') >='2026-05'))

##### Filtering any row where pick up date is smaller than 2026-04

In [ ]:
green_taxi_data = green_taxi_data.where(~(date_format('lpep_pickup_datetime',format = 'yyyy-MM') <'2026-04'))

##### Filtering rows where passenger count  == 0

In [ ]:
green_taxi_data = green_taxi_data.filter(col('passenger_count') != 0)

##### Filtering rows for negative fare amount

In [ ]:
green_taxi_data = green_taxi_data.filter(col('fare_amount') > 0.0)

##### checking if any trip with negative distance

In [ ]:
green_taxi_data.filter(col('trip_distance') < 0.0).show()

##### Checking if any trip with pickup time later than drop off time

In [ ]:
green_taxi_data.where(col('lpep_pickup_datetime') > col('lpep_dropoff_datetime')).count()

#### Log Count after validation

In [ ]:
logger.info(f"After Validation and Filtering Count : {green_taxi_data.coalesce(1).count()}")

#### Derive trip_duration_minutes

In [ ]:

green_taxi_data = green_taxi_data.withColumn('trip_duration_minutes', timestamp_diff('minute',col('lpep_pickup_datetime'),col('lpep_dropoff_datetime')))

#### Calculating avg_speed_mph

In [ ]:
green_taxi_data = green_taxi_data.withColumn('avg_speed_mph',round(when(col('trip_duration_minutes') == 0, 0).\
                                            otherwise(col('trip_distance')/(col('trip_duration_minutes')/60)),2))

#### Add source file name and ingestion timestamp

In [ ]:
green_taxi_data = green_taxi_data.withColumns({'source_file': lit('green_taxi_trip_data.parquet'),
                              'ingested_at_timestamp' : current_timestamp()})

In [ ]:
s3_bucket = os.environ["S3_BUCKET"]

green_taxi_data \
    .withColumn('pickup_date', date_format(col('lpep_pickup_datetime'), 'yyyy-MM-dd')) \
    .write \
    .option('header', True) \
    .partitionBy('pickup_date') \
    .mode('overwrite') \
    .parquet(f's3a://{s3_bucket}/green')

In [ ]:
logger.info(f"No.of records written to S3 Bucket : {green_taxi_data.count()}")

In [ ]:
spark.stop()
del spark

#### Read Paritioned Data

In [ ]:
read_partitioned_data = spark.read.parquet('s3a://nyc-tlc-pipeline/green')

In [ ]:
read_partitioned_data.printSchema()